# Esquema: clasificación binaria (MVP manual)

Target **texto** → 0/1 manual. Varios clasificadores en **`make_pipeline(StandardScaler, modelo)`**.

| Modelos en pipeline | |
|---------------------|---|
| Regresión logística, KNN, árbol, bosque aleatorio, SVM | |

Siguiente: [07.b binaria](../07.b-ejemplos-supervisados/02-clasificacion-binaria.ipynb).


## 1. CSV, tipos y faltantes

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("data/datos_spam.csv")
print(df.dtypes)
print("\nFaltantes:\n", df.isna().sum())
display(df)


Palabras_En_Texto    float64
Etiqueta_Texto        object
dtype: object

Faltantes:
 Palabras_En_Texto    1
Etiqueta_Texto       1
dtype: int64


,Palabras_En_Texto,Etiqueta_Texto
0,3.0,no
1,8.0,no
2,12.0,no
3,25.0,si
4,30.0,si
5,45.0,si
6,5.0,no
7,NaN,si
8,22.0,si
9,7.0,no


## 2. Target texto → 0/1

In [2]:
MAPA_BINARIO = {"no": 0, "si": 1}
df = df.dropna(subset=["Etiqueta_Texto"]).copy()
y = df["Etiqueta_Texto"].str.strip().str.lower().map(MAPA_BINARIO).astype(int)


## 3. Feature numérica

In [3]:
palabras = df["Palabras_En_Texto"].astype(float)
X = pd.DataFrame({"Palabras_En_Texto": palabras.fillna(palabras.median())})


## 4. Split train / val / test (estratificado)


In [4]:
def split_train_val_test(X, y, test_size, val_size, random_state, stratify=False):
    """Divide en train, validación y test (dos llamadas a train_test_split).

    - test_size: fracción del total para test (hold-out final).
    - val_size: fracción de train+val → validación.
    Con test_size=0.2 y val_size=0.25 → ~60 % train, ~20 % val, ~20 % test.
    """
    kw = dict(test_size=test_size, random_state=random_state)
    if stratify:
        kw["stratify"] = y
    X_tv, X_test, y_tv, y_test = train_test_split(X, y, **kw)
    kw2 = dict(test_size=val_size, random_state=random_state)
    if stratify:
        kw2["stratify"] = y_tv
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, **kw2)
    return X_train, X_val, X_test, y_train, y_val, y_test


RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.25

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=True
)
print(f"Tamaños → train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")


Tamaños → train: 6 | val: 2 | test: 2


## 5. Entrenar modelos en Pipeline

Bucle sobre `build_models()`: **fit** en train; guardar pipelines en `pipelines` y predicciones en `predicciones_test`.

Solo **`fit` en train**; predicciones y métricas van en el apartado de análisis.


In [5]:
def build_models():
    """Misma lista que 07.b (comenta entradas para excluir modelos)."""
    from sklearn.ensemble import (
        GradientBoostingClassifier,
        HistGradientBoostingClassifier,
        RandomForestClassifier,
    )
    from sklearn.linear_model import LogisticRegression, SGDClassifier
    from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.tree import DecisionTreeClassifier
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier

    return {
        "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "SGDClassifier": SGDClassifier(
            loss="log_loss",
            max_iter=2000,
            tol=1e-3,
            random_state=RANDOM_STATE,
        ),
        "SVC": SVC(random_state=RANDOM_STATE),
        "OneVsOneClassifier": OneVsOneClassifier(SVC(random_state=RANDOM_STATE)),
        "OneVsRestClassifier": OneVsRestClassifier(SVC(random_state=RANDOM_STATE)),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "DecisionTree": DecisionTreeClassifier(
            criterion="gini",
            splitter="best",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=None,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            random_state=RANDOM_STATE,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            criterion="gini",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features="sqrt",
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(
            random_state=RANDOM_STATE,
            verbosity=0,
            n_estimators=100,
            eval_metric="logloss",
            n_jobs=-1,
        ),
        "CatBoost": CatBoostClassifier(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
        ),
    }


RANDOM_STATE = 42
MODELS = build_models()


pipelines = {}
for nombre, modelo in MODELS.items():
    pipe = make_pipeline(StandardScaler(), modelo)
    pipe.fit(X_train, y_train)
    pipelines[nombre] = pipe


## 6. Análisis comparativo

Tabla por **accuracy en val**; reporte en **test** del ganador.

Predicciones en **val** y **test**, tabla comparativa y elección del ganador por **val**.


In [6]:
filas = []
predicciones_test = {}
for nombre, pipe in pipelines.items():
    pred_val = pipe.predict(X_val)
    pred_test = pipe.predict(X_test)
    acc_val = accuracy_score(y_val, pred_val)
    acc_test = accuracy_score(y_test, pred_test)
    predicciones_test[nombre] = pred_test
    filas.append(
        {"modelo": nombre, "accuracy_val": acc_val, "accuracy_test": acc_test}
    )

tabla = pd.DataFrame(filas).sort_values("accuracy_val", ascending=False)
display(tabla.round(4))

mejor = tabla.iloc[0]
mejor_nombre = mejor["modelo"]
print(f"\nMejor accuracy en val: {mejor_nombre} ({mejor['accuracy_val']:.2f})")
print(f"Accuracy en test del ganador: {mejor['accuracy_test']:.2f}")
print(classification_report(
    y_test, predicciones_test[mejor_nombre], target_names=["No spam (0)", "Spam (1)"]
))


,modelo,accuracy_val,accuracy_test
0,LogisticRegression,1.0,1.0
1,SGDClassifier,1.0,1.0
2,SVC,1.0,1.0
3,OneVsOneClassifier,1.0,1.0
4,OneVsRestClassifier,1.0,1.0
6,DecisionTree,1.0,1.0
7,RandomForest,1.0,1.0
8,GradientBoosting,1.0,1.0
11,CatBoost,1.0,1.0
5,KNN,0.5,0.5



Mejor accuracy en val: LogisticRegression (1.00)
Accuracy en test del ganador: 1.00
              precision    recall  f1-score   support

 No spam (0)       1.00      1.00      1.00         1
    Spam (1)       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2

